# 面试问题：RLHF 中的 LLM PPO 怎样实现？Token-level KL、GAE、clip 和 mask 分别做什么？

**一句话回答**：把 prompt 视为条件、response token 视为动作序列；Reward Model 通常给序列末端奖励，同时每个生成 token 加相对 reference policy 的 KL 惩罚。Value 估计未来回报，GAE 构造 advantage，PPO 用 old policy 的固定 log-prob 计算 clipped ratio，并且所有损失只落在有效 response token 上。

本 Notebook 用 PyTorch 基础张量和手写 SGD 实现 trajectory contract、KL shaping、GAE、policy/value clip、padding mask、更新与 early-stop。它是机制实验，不是完整分布式 RLHF trainer。重点是把生成时状态、训练时旧策略和发布评测连成可审计闭环，并展示一个 mask 错位为何会污染整批梯度与训练证据。


In [ ]:
from dataclasses import dataclass
import math
import torch

SEED133=13301; torch.manual_seed(SEED133)
assert SEED133==13301
assert torch.__version__
assert torch.isfinite(torch.randn(3)).all()


## 1. Rollout 必须绑定生成时的版本和 action mask

一条样本至少保存 prompt/response token、有效 response mask、old log-prob、reference log-prob、old value、终止原因和 reward model 版本。PPO epoch 中 `old_logp` 必须冻结；重新用当前 policy 计算会让 importance ratio 恒接近 1，clip 失去意义。


In [ ]:
@dataclass
class Rollout133:
    tokens:torch.Tensor; response_mask:torch.Tensor; old_logp:torch.Tensor; ref_logp:torch.Tensor; old_value:torch.Tensor; terminal_reward:float
    def __post_init__(self):
        n=len(self.tokens)
        if any(len(x)!=n for x in (self.response_mask,self.old_logp,self.ref_logp,self.old_value)): raise ValueError("length_contract")
ro133=Rollout133(torch.tensor([8,9,2,0]),torch.tensor([1.,1.,1.,0.]),torch.tensor([-.5,-.4,-.2,0.]),torch.tensor([-.45,-.35,-.25,0.]),torch.zeros(4),1.0)
assert ro133.response_mask.sum()==3
assert ro133.tokens.shape==ro133.old_logp.shape
try: Rollout133(torch.tensor([1]),torch.ones(2),torch.ones(1),torch.ones(1),torch.ones(1),0); raise AssertionError("bad rollout")
except ValueError as e: assert str(e)=="length_contract"


## 2. 序列奖励被塑形成逐 token reward

常见采样估计在每个 action 上加入 `-β(logπ-logπ_ref)`，最后一个有效 token 再加 RM/outcome reward。它不是完整词表 KL 的无偏单样本真值，但可作为 rollout shaping；监控时还应独立计算分布 KL 或稳定近似，避免符号写反导致策略远离 reference。


In [ ]:
def shaped_rewards133(logp,ref_logp,mask,terminal,beta):
    r=-beta*(logp-ref_logp)*mask; last=int(torch.nonzero(mask)[-1]); r[last]+=terminal; return r
rewards133=shaped_rewards133(ro133.old_logp,ro133.ref_logp,ro133.response_mask,1.0,.1)
assert torch.allclose(rewards133,torch.tensor([.005,.005,.995,0.]))
assert math.isclose(float(rewards133.sum()),1.005,rel_tol=1e-6)
assert rewards133[-1]==0


## 3. GAE 在 bias 与 variance 间折中

`δ_t=r_t+γV_{t+1}-V_t`，再反向累计 `A_t=δ_t+γλA_{t+1}`。EOS/terminal 后 bootstrap 为零；若只是长度截断而环境未终止，则应保留最后 value。这里对 response mask 反向计算，并让 padding 的 advantage/return 为零。


In [ ]:
def gae133(rewards,values,mask,gamma=.99,lam=.95):
    adv=torch.zeros_like(rewards); carry=torch.tensor(0.); valid=int(mask.sum())
    for t in reversed(range(valid)):
        next_v=values[t+1] if t+1<valid else torch.tensor(0.)
        delta=rewards[t]+gamma*next_v-values[t]; carry=delta+gamma*lam*carry; adv[t]=carry
    return adv,(adv+values)*mask
adv133,ret133=gae133(rewards133,ro133.old_value,ro133.response_mask)
assert adv133[2]>adv133[1]>adv133[0]>0
assert ret133[-1]==0
assert torch.allclose(adv133,ret133)


## 4. PPO clip 限制单批数据上的策略跃迁

`ratio=exp(new_logp-old_logp)`，目标是 `min(ratio*A, clip(ratio,1-ε,1+ε)*A)`。对负 advantage，取 min 的方向同样重要，不能把 ratio 先简单截断再总是使用。loss 只在 response mask 上求平均，prompt token 不参与策略梯度。


In [ ]:
def policy_loss133(new_logp,old_logp,adv,mask,eps=.2):
    ratio=torch.exp(new_logp-old_logp); unclipped=ratio*adv; clipped=ratio.clamp(1-eps,1+eps)*adv
    return -(torch.minimum(unclipped,clipped)*mask).sum()/mask.sum(),ratio
newp133=ro133.old_logp+torch.tensor([.4,-.4,.0,9.]); ploss133,ratio133=policy_loss133(newp133,ro133.old_logp,torch.tensor([1.,-1.,.5,2.]),ro133.response_mask)
assert ratio133[0]>1.2 and ratio133[1]<.8
assert torch.isfinite(ploss133)
assert ratio133[-1]>1000 and ro133.response_mask[-1]==0


## 5. Value clip 防止 critic 在同一 rollout 上过拟合跳变

Value loss 比较未裁剪预测与 `old_value±ε_v` 的预测误差，逐 token 取较大者作为保守目标。policy/value 可共享 backbone，但 mask、loss coefficient 和梯度尺度要分别记录；critic 失真会直接污染 GAE。


In [ ]:
def value_loss133(new_v,old_v,returns,mask,eps=.2):
    clipped=old_v+(new_v-old_v).clamp(-eps,eps); e1=(new_v-returns).square(); e2=(clipped-returns).square()
    return .5*(torch.maximum(e1,e2)*mask).sum()/mask.sum()
newv133=torch.tensor([2.,.4,.8,100.]); vloss133=value_loss133(newv133,ro133.old_value,ret133,ro133.response_mask)
assert vloss133>0
assert torch.isfinite(vloss133)
assert value_loss133(ret133,ret133,ret133,ro133.response_mask)==0


## 6. Log-prob 要按生成 token gather，并严格处理 shift/padding

logits 位置 `t` 预测 token `t+1`；训练数据必须保存与 rollout 完全一致的 tokenizer、temperature 和 stop 规则。下面手写 log-softmax 与 gather，不调用封装的交叉熵，并证明 padding logit 再离谱也不会进入 masked 平均。


In [ ]:
def log_softmax133(x): return x-torch.logsumexp(x,dim=-1,keepdim=True)
def chosen_logp133(logits,actions): return log_softmax133(logits).gather(-1,actions[:,None]).squeeze(-1)
logits133=torch.tensor([[1.,2.,0.],[3.,0.,1.],[0.,0.,0.],[100.,-100.,0.]])
picked133=chosen_logp133(logits133,torch.tensor([1,0,2,1]))
assert picked133.shape==(4,)
assert picked133[1]>picked133[0]
assert torch.isfinite((picked133[:3]*ro133.response_mask[:3]).sum())


## 7. 每轮只复用有限 epoch，并按 KL 提前停止

rollout 是旧策略分布产生的；重复优化太多会让 ratio 和 KL 失控。打乱 minibatch 仍要按 prompt group/长度正确组装，达到 target KL、overflow 或异常 reward 时停止，并丢弃对应更新。示例用一个可学习 logit 向量和手写 SGD 展示梯度路径。


In [ ]:
theta133=torch.nn.Parameter(torch.tensor([-.45,-.45,-.25,0.])); before133=theta133.detach().clone()
loss133,_=policy_loss133(theta133,ro133.old_logp,adv133.detach(),ro133.response_mask); loss133.backward()
with torch.no_grad(): theta133-=.05*theta133.grad
sampled_kl133=float(((theta133.detach()-ro133.ref_logp)*ro133.response_mask).sum()/ro133.response_mask.sum())
assert not torch.allclose(theta133,before133)
assert torch.isfinite(theta133).all()
assert isinstance(sampled_kl133,float)


## 8. Reward 上升不等于对齐成功

发布门禁同时看人类/规则任务成功率、RM reward、真实 KL、长度、拒答、安全 slice 和能力回归；检查 reward hacking、模式坍缩与 reference 漂移。训练日志按 prompt ID 保留 policy/ref/RM 版本和采样参数，但不默认落盘敏感原文。


In [ ]:
eval133=[{"rm":.9,"human":1,"len":20},{"rm":.95,"human":0,"len":200},{"rm":.7,"human":1,"len":18}]
rm_mean133=sum(x["rm"] for x in eval133)/len(eval133); human133=sum(x["human"] for x in eval133)/len(eval133)
assert rm_mean133>.8
assert human133<rm_mean133
assert max(x["len"] for x in eval133)>10*min(x["len"] for x in eval133)


## 面试总结

主线是：**rollout 固化 old/ref/value 与版本 → response-token mask → 逐 token KL shaping、末端 RM reward → terminal-aware GAE → PPO policy clip + value clip → 正确 shift/gather → 有限 epoch 与 target-KL stop → 全局 overflow 原子跳步 → 用人工质量、安全、长度和能力回归识别 reward hacking**。PPO 的难点主要在数据/版本/掩码合同，而不只是一个公式。

延伸阅读：[InstructGPT](https://arxiv.org/abs/2203.02155)、[PPO](https://arxiv.org/abs/1707.06347)、[Learning to Summarize from Human Feedback](https://arxiv.org/abs/2009.01325)。
